In [1]:
import pandas as pd
import gc

## read the fines.csv that was saved in the previous exercise

In [2]:
df = pd.read_csv('../data/fines.csv')

In [3]:
df.head()

,CarNumber,Refund,Fines,Make,Model,Year
0,Y163O8161RUS,2,3200.0,Ford,Focus,1989
1,E432XX77RUS,1,6500.0,Toyota,Camry,1995
2,7184TT36RUS,1,2100.0,Ford,Focus,1984
3,X582HE161RUS,2,2000.0,Ford,Focus,2015
4,92918M178RUS,1,5700.0,Ford,Focus,2014


## iterations

1. loop

In [4]:
def calculate_fines_by_loop(df):
    new_column = []
    for i in range(0, len(df)):
        new_column.append(df.iloc[i].Fines / df.iloc[i].Refund * df.iloc[i].Year)
    df['Results'] = new_column
    return df

In [5]:
%%timeit
calculate_fines_by_loop(df.copy())

87 ms ± 11.2 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


2. iterrows()

In [6]:
def calculate_fines_by_iterrows(df):
    new_column = []
    for i,row in df.iterrows():
        new_column.append(row['Fines'] / row['Refund'] * row['Year'])
    df['Results'] = new_column
    return df    

In [7]:
%%timeit
calculate_fines_by_iterrows(df.copy())

20.9 ms ± 1.91 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


3. apply()

In [8]:
df_for_apply = df.copy()

In [9]:
%%timeit
df_for_apply['Results'] = df_for_apply.apply(lambda x: x.Fines / x.Refund * x.Year, axis=1)

11.5 ms ± 469 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


4. Series

In [10]:
df_for_Series = df.copy()

In [11]:
%%timeit
df_for_Series['Results'] = df_for_Series.Fines / df_for_Series.Refund * df_for_Series.Year

155 μs ± 13.7 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


5. Series with method values

In [12]:
df_for_values = df.copy()

In [13]:
%%timeit
df_for_values['Results'] = df_for_Series.Fines.values / df_for_Series.Refund.values * df_for_Series.Year.values

30.2 μs ± 3.83 μs per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


In [14]:
df['strange'] = df_for_Series.Fines.values * df_for_Series.Refund.values / df_for_Series.Year.values

## indexing

In [15]:
df_index = df.copy()

In [16]:
%%timeit
df_index.loc[df_index.CarNumber == 'O136HO197RUS']

180 μs ± 5.23 μs per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


In [17]:
df_index.set_index('CarNumber', inplace=True)
df_index.head()

,Refund,Fines,Make,Model,Year,strange
CarNumber,,,,,,
Y163O8161RUS,2,3200.0,Ford,Focus,1989,3.217697
E432XX77RUS,1,6500.0,Toyota,Camry,1995,3.258145
7184TT36RUS,1,2100.0,Ford,Focus,1984,1.058468
X582HE161RUS,2,2000.0,Ford,Focus,2015,1.985112
92918M178RUS,1,5700.0,Ford,Focus,2014,2.830189


In [18]:
%%timeit
df_index.loc['O136HO197RUS']

89.3 μs ± 4.15 μs per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


## downcasting

In [19]:
df.info(memory_usage='deep')

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 930 entries, 0 to 929
Data columns (total 7 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   CarNumber  930 non-null    object 
 1   Refund     930 non-null    int64  
 2   Fines      930 non-null    float64
 3   Make       930 non-null    object 
 4   Model      919 non-null    object 
 5   Year       930 non-null    int64  
 6   strange    930 non-null    float64
dtypes: float64(2), int64(2), object(3)
memory usage: 182.1 KB


In [20]:
optimized = df.copy()

In [21]:
optimized['Fines'] = pd.to_numeric(optimized['Fines'], downcast='float')
optimized['strange'] = pd.to_numeric(optimized['strange'], downcast='float')
optimized['Refund'] = pd.to_numeric(optimized['Refund'], downcast='integer')
optimized['Year'] = pd.to_numeric(optimized['Year'], downcast='integer')

In [22]:
optimized.info(memory_usage='deep')

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 930 entries, 0 to 929
Data columns (total 7 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   CarNumber  930 non-null    object 
 1   Refund     930 non-null    int8   
 2   Fines      930 non-null    float32
 3   Make       930 non-null    object 
 4   Model      919 non-null    object 
 5   Year       930 non-null    int16  
 6   strange    930 non-null    float32
dtypes: float32(2), int16(1), int8(1), object(3)
memory usage: 163.0 KB


## categories

In [23]:
optimized['Make'] = optimized['Make'].astype('category')
optimized['Model'] = optimized['Model'].astype('category')
optimized['CarNumber'] = optimized['CarNumber'].astype('category')

In [24]:
optimized.info(memory_usage='deep')

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 930 entries, 0 to 929
Data columns (total 7 columns):
 #   Column     Non-Null Count  Dtype   
---  ------     --------------  -----   
 0   CarNumber  930 non-null    category
 1   Refund     930 non-null    int8    
 2   Fines      930 non-null    float32 
 3   Make       930 non-null    category
 4   Model      919 non-null    category
 5   Year       930 non-null    int16   
 6   strange    930 non-null    float32 
dtypes: category(3), float32(2), int16(1), int8(1)
memory usage: 63.3 KB


## memory clean

In [25]:
%reset_selective df
gc.collect()

7

In [26]:
optimized.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 930 entries, 0 to 929
Data columns (total 7 columns):
 #   Column     Non-Null Count  Dtype   
---  ------     --------------  -----   
 0   CarNumber  930 non-null    category
 1   Refund     930 non-null    int8    
 2   Fines      930 non-null    float32 
 3   Make       930 non-null    category
 4   Model      919 non-null    category
 5   Year       930 non-null    int16   
 6   strange    930 non-null    float32 
dtypes: category(3), float32(2), int16(1), int8(1)
memory usage: 34.8 KB


In [29]:
df

NameError: name 'df' is not defined